# Projeto Final — Análise exploratória do crédito rural brasileiro (2025)

**Disciplina:** Técnicas de Programação I — DS-PY-004  
**Ferramentas:** Python, Pandas e NumPy  
**Recorte:** operações de crédito rural emitidas em 2025, em todas as unidades da Federação.

## Etapa 1 — Apresentação e compreensão da base

### 1.1 — Origem dos dados e objetivo

Os dados têm origem no **Sistema de Operações do Crédito Rural e do Proagro (SICOR)**, mantido pelo **Banco Central do Brasil (BCB)**. Os arquivos utilizados foram extraídos por meio da **Base dos Dados / Google BigQuery**, a partir do conjunto `basedosdados.br_bcb_sicor`.

- [Fonte oficial — Banco Central do Brasil](https://www.bcb.gov.br/estabilidadefinanceira/micrrural)
- [Conjunto de dados — Base dos Dados](https://basedosdados.org/dataset/544c9d22-97b7-479a-8eca-94762840b465)

**Objetivo:** investigar a distribuição, as diferenças de valor, a concentração e a variação mensal das operações de crédito rural de 2025, usando evidências extraídas dos dados.

**Atenção:** o recorte considera **ano de emissão**, não necessariamente o ano de desembolso, execução do empreendimento ou vencimento. As conclusões devem respeitar essa delimitação.

### 1.2 — Perguntas de pesquisa (definidas antes da análise)

1. **Distribuição geográfica:** Como o volume financeiro e a quantidade de operações de crédito rural se distribuem entre os estados brasileiros?
2. **Categoria do produtor:** Existem diferenças significativas no valor médio das operações entre as categorias de produtores rurais?
3. **Finalidade do crédito:** Como o volume e o valor médio das operações variam entre as finalidades de crédito rural?
4. **Concentração:** Qual é o grau de concentração dos recursos de crédito rural entre os estados e as atividades agropecuárias?
5. **Distribuição mensal:** Como a distribuição mensal do crédito rural varia entre as principais atividades agropecuárias?

**Planejamento metodológico:** usaremos totais e contagens por UF; valor médio por categoria e finalidade; participação percentual das UFs/atividades no valor financiado; e séries mensais por atividade. A interpretação dependerá da consistência dos registros e das chaves.

Estas são **perguntas de investigação**, não conclusões antecipadas.

### 1.3 — Arquivos utilizados e relacionamentos

| Arquivo | Papel na análise | Principais campos |
|---|---|---|
| `operacao_2025.csv` | Base principal de registros de crédito rural | `ano_emissao`, `mes_emissao`, `sigla_uf`, `id_referencia_bacen`, `numero_ordem`, `id_categoria_emitente`, `id_empreendimento`, `valor_parcela_credito` |
| `empreendimento_2025.csv` | Características dos empreendimentos vinculados às operações | `id_empreendimento`, `finalidade`, `atividade`, `modalidade`, `produto` |
| `dicionario.csv` | Correspondência entre códigos e descrições de campos | `id_tabela`, `nome_coluna`, `chave`, `valor`, `cobertura_temporal` |

Conforme no `README.md`, a base de operações contém **2.442.343 linhas e 15 colunas**, e a de empreendimentos contém **1.050 linhas e 10 colunas**. As dimensões serão conferidas novamente no código, sem assumir que os arquivos permaneçam inalterados.


**Cuidado:** `dicionario.csv` reúne diferentes variáveis e coberturas temporais; não devemos associar códigos de categorias sem filtrar o campo correto e conferir a correspondência.

### 1.4 — Importação das bibliotecas e leitura dos arquivos

**O que será feito:** Importar Pandas e NumPy e carregar as três bases CSV em DataFrames chamados `operacao`, `dicionario` e `empreendimento`.

**Como será feito:** Importaremos `pandas` e `numpy` com `import` e carregaremos cada CSV com `pd.read_csv()`, mantendo os três arquivos em DataFrames separados.

**Por que será feito:** Precisamos disponibilizar as tabelas no Python antes de inspecionar sua estrutura. O Pandas será usado para manipular tabelas; o NumPy será utilizado em cálculos posteriores. **Nesta célula não há limpeza nem transformação dos registros.**

In [2]:
import pandas as pd
import numpy as np

operacao = pd.read_csv("dados/operacao_2025.csv")
dicionario = pd.read_csv("dados/dicionario.csv")
empreendimento = pd.read_csv("dados/empreendimento_2025.csv")

### 1.5 — Quantidade de registros e colunas

**O que será feito:** Exibir a quantidade de linhas e colunas de cada DataFrame utilizando `.shape`.

**Como será feito:** Usaremos `DataFrame.shape[0]` para contar linhas e `DataFrame.shape[1]` para contar colunas de cada tabela.

**Por que será feito:** Conhecer o tamanho de cada arquivo ajuda a dimensionar a análise e verificar o requisito mínimo de linhas e colunas do projeto. A contagem não informa, por si só, quantas operações únicas existem.

In [2]:
print(f"A base principal operacao_2025.csv possui {operacao.shape[0]} registros e {operacao.shape[1]} colunas.")
print(f"A base segundaria empreendimento_2025.csv possui {empreendimento.shape[0]} registros e {empreendimento.shape[1]} colunas.")
print(f"A base dicionario.csv possui {dicionario.shape[0]} registros e {dicionario.shape[1]} colunas.")

A base principal operacao_2025.csv possui 2442343 registros e 15 colunas.
A base segundaria empreendimento_2025.csv possui 1050 registros e 10 colunas.
A base dicionario.csv possui 52 registros e 5 colunas.


### 1.6 — Listagem das colunas das três tabelas

**O que será feito:** Mostrar os nomes de todas as colunas por meio de `.columns.tolist()`.

**Como será feito:** Usaremos `DataFrame.columns.tolist()` para obter a lista dos nomes das colunas de cada tabela.

**Por que será feito:** Reconhecer os campos disponíveis em cada arquivo é necessário para localizar variáveis relevantes às perguntas de pesquisa e identificar possíveis chaves para relacionar tabelas.

In [3]:
print(f"O nomes das colunas da base principal operacao_2025.csv e {operacao.columns.tolist()}")
print(f"O nomes das colunas da segundaria empreendimento_2025.csv e {empreendimento.columns.tolist()}")
print(f"O nomes das colunas da base dicionario.csvcsv e {dicionario.columns.tolist()}")


O nomes das colunas da base principal operacao_2025.csv e ['ano_emissao', 'mes_emissao', 'data_emissao', 'data_vencimento', 'sigla_uf', 'id_referencia_bacen', 'numero_ordem', 'id_categoria_emitente', 'id_empreendimento', 'id_fonte_recurso', 'id_instrumento_credito', 'valor_parcela_credito', 'valor_quantidade_itens_financiados', 'valor_recurso_proprio', 'valor_receita_bruta_esperada']
O nomes das colunas da segundaria empreendimento_2025.csv e ['id_empreendimento', 'data_inicio', 'data_fim', 'finalidade', 'atividade', 'modalidade', 'produto', 'variedade', 'zoneamento', 'unidade_medida']
O nomes das colunas da base dicionario.csvcsv e ['id_tabela', 'nome_coluna', 'chave', 'cobertura_temporal', 'valor']


### 1.7 — Numeração das colunas de operações

**O que será feito:** Percorrer e numerar os nomes das colunas da tabela `operacao`.

**Como será feito:** Usaremos `enumerate(operacao.columns, start=1)` e `print()` para numerar e exibir as colunas.

**Por que será feito:** A numeração facilita a leitura e a documentação dos campos da base principal, sobretudo quando existem muitas colunas.

In [4]:
print("O nomes das colunas da base principal operacao_2025.csv e:")
for numero, coluna in enumerate(operacao.columns, start=1):    
    print(f"{numero}. {coluna}")

O nomes das colunas da base principal operacao_2025.csv e:
1. ano_emissao
2. mes_emissao
3. data_emissao
4. data_vencimento
5. sigla_uf
6. id_referencia_bacen
7. numero_ordem
8. id_categoria_emitente
9. id_empreendimento
10. id_fonte_recurso
11. id_instrumento_credito
12. valor_parcela_credito
13. valor_quantidade_itens_financiados
14. valor_recurso_proprio
15. valor_receita_bruta_esperada


### 1.8 — Numeração das colunas de empreendimentos

**O que será feito:** Percorrer e numerar os nomes das colunas da tabela `empreendimento`.

**Como será feito:** Usaremos `enumerate(empreendimento.columns, start=1)` e `print()` para numerar e exibir as colunas.

**Por que será feito:** Conhecer os campos da tabela auxiliar ajudará a avaliar mais tarde quais informações podem complementar a base de operações.

In [5]:
print("O nomes das colunas da base principal empreendimento_2025.csv e:")
for numero, coluna in enumerate(empreendimento.columns, start=1):    
    print(f"{numero}. {coluna}")

O nomes das colunas da base principal empreendimento_2025.csv e:
1. id_empreendimento
2. data_inicio
3. data_fim
4. finalidade
5. atividade
6. modalidade
7. produto
8. variedade
9. zoneamento
10. unidade_medida


### 1.9 — Numeração das colunas do dicionário

**O que será feito:** listar e numerar as colunas do arquivo `dicionario.csv`.

**Como será feito:** Usaremos `enumerate(dicionario.columns, start=1)` e `print()` para numerar e exibir as colunas.

**Por que será feito:** confirmar a estrutura da tabela auxiliar, que será utilizada para interpretar códigos presentes na base de operações. Esse procedimento é apenas uma inspeção, sem alterar os dados.

In [10]:
print("O nomes das colunas da base dicionario.csv e:")
for numero, coluna in enumerate(dicionario.columns, start=1):
    print(f"{numero}. {coluna}")

O nomes das colunas da base dicionario.csv e:
1. id_tabela
2. nome_coluna
3. chave
4. cobertura_temporal
5. valor


### 1.10 — Primeiros registros das operações

**O que será feito:** Exibir as cinco primeiras linhas de `operacao` com `.head(5)`.

**Como será feito:** Usaremos `operacao.head(5)` para visualizar as cinco primeiras linhas da tabela de operações.

**Por que será feito:** A amostra inicial permite visualizar a estrutura dos registros e exemplos dos valores. **Cinco linhas não são suficientes para concluir se a base possui erros ou dados faltantes.**

In [121]:
operacao.head(5)

,ano_emissao,mes_emissao,data_emissao,data_vencimento,sigla_uf,id_referencia_bacen,numero_ordem,id_categoria_emitente,id_empreendimento,id_fonte_recurso,id_instrumento_credito,valor_parcela_credito,valor_quantidade_itens_financiados,valor_recurso_proprio,valor_receita_bruta_esperada
0,2025,11,2025-11-12,2026-11-11,AL,521092090,1,2222,12010040000041,201,1,30000.00,NaN,NaN,126500.0
1,2025,4,2025-04-08,2026-11-30,AL,519741649,1,2222,12010040000041,201,10,93225.68,NaN,0.0,408135.0
2,2025,3,2025-03-13,2026-11-30,AL,519609806,1,2222,12010040000041,300,10,86845.56,NaN,0.0,394530.5
3,2025,4,2025-04-08,2026-11-30,AL,519741594,1,2222,12010040000041,201,10,33960.78,NaN,0.0,145762.5
4,2025,2,2025-02-18,2026-12-10,AL,519516592,1,2222,12010040000041,201,10,31230.17,NaN,0.0,141401.0


### 1.11 — Primeiros registros dos empreendimentos

**O que será feito:** Exibir as cinco primeiras linhas de `empreendimento`.

**Como será feito:** Usaremos `empreendimento.head(5)` para visualizar as cinco primeiras linhas da tabela de empreendimentos.

**Por que será feito:** Essa observação inicial ajuda a reconhecer os códigos e as descrições que poderão ser usados para interpretar as operações.

In [13]:
empreendimento.head(5)

,id_empreendimento,data_inicio,data_fim,finalidade,atividade,modalidade,produto,variedade,zoneamento,unidade_medida
0,23271400000002,1984-01-01,NaN,Investimento,Pecuário(a),AQUISIÇÃO DE ANIMAIS,BÚFALOS (BUBALINOS),NÃO SE APLICA,Não zoneado,Unidade
1,23271300193002,1984-01-01,NaN,Investimento,Pecuário(a),AQUISIÇÃO DE ANIMAIS,BOVINOS,LEITE - MATRIZES (VACAS E CRIAS FÊMEAS PARA P...,Não zoneado,Unidade
2,23276760000002,1984-01-01,NaN,Investimento,Pecuário(a),AQUISIÇÃO DE ANIMAIS,SUÍNOS,NÃO SE APLICA,Não zoneado,Unidade
3,23275580000002,1984-01-01,NaN,Investimento,Pecuário(a),AQUISIÇÃO DE ANIMAIS,OVINOS,NÃO SE APLICA,Não zoneado,Unidade
4,23270820015002,1984-01-01,NaN,Investimento,Pecuário(a),AQUISIÇÃO DE ANIMAIS,APICULTURA,"ABRANGE COMPRA DE ABELHAS, ISOLADAMENTE OU EM ...",Não zoneado,Unidade


### 1.12 — Primeiros registros do dicionário

**O que será feito:** Exibir os quinze primeiros registros de `dicionario`.

**Como será feito:** Usaremos `dicionario.head(15)` para visualizar quinze exemplos de códigos e descrições.

**Por que será feito:** O dicionário pode explicar códigos presentes nas tabelas. Inspecionar seus registros permite entender como os códigos e seus significados estão organizados.

In [3]:
dicionario.head(15)

,id_tabela,nome_coluna,chave,cobertura_temporal,valor
0,operacao,id_categoria_emitente,2222,2013(1),Pequeno Produtor Rural
1,operacao,id_categoria_emitente,3333,2013(1),Médio Produtor Rural
2,operacao,id_categoria_emitente,4444,2013(1),Grande Produtor Rural
3,operacao,id_categoria_emitente,5555,2012(1)2013,Todas Categorias de Produtores Rurais
4,operacao,id_categoria_emitente,7777,2013(1)2018,Repasse a Cooperativa Singular de Crédito
5,operacao,id_categoria_emitente,8888,2013(1)2018,Repasse a Cooperativa Central de Crédito
6,operacao,id_fonte_recurso,100,2011(1)2099,TESOURO NACIONAL
7,operacao,id_fonte_recurso,201,2011(1)2099,OBRIGATÓRIOS - MCR 6.2
8,operacao,id_fonte_recurso,202,2019(1)2019,EXIGIBILIDADE ADICIONAL DOS RECURSOS À VISTA
9,operacao,id_fonte_recurso,222,2022(1)2023,Exigibilidade Adicional dos Recursos à Vista -...


### 1.13 — Cópia de trabalho da tabela de operações

**O que será feito:** Criar `operacao_traduzida` como cópia de `operacao`, utilizando `.copy()`.

**Como será feito:** Usaremos `operacao.copy()` para criar o DataFrame `operacao_traduzida`, sem modificar a tabela original.

**Por que será feito:** Uma cópia separada permite preparar transformações futuras sem alterar o DataFrame original. **Esta célula apenas cria a cópia: ainda não traduz categorias nem limpa dados.**

In [5]:
operacao_traduzida = operacao.copy()

### 1.14 — Seleção das descrições da categoria do emitente

**O que será feito:** Filtrar o dicionário, mantendo as linhas em que `nome_coluna` é igual a `id_categoria_emitente`, e mostrar o resultado.

**Como será feito:** Aplicaremos um filtro booleano com `dicionario["nome_coluna"] == "id_categoria_emitente"` e mostraremos a seleção com `display()`.

**Por que será feito:** Essa seleção prepara a futura interpretação dos códigos da categoria de produtor. Antes de um `merge`, ainda será necessário verificar as chaves, a unicidade dos códigos e a quantidade de correspondências.

In [6]:
dicionario_categoria = dicionario[dicionario["nome_coluna"] == "id_categoria_emitente"]
display(dicionario_categoria)

,id_tabela,nome_coluna,chave,cobertura_temporal,valor
0,operacao,id_categoria_emitente,2222,2013(1),Pequeno Produtor Rural
1,operacao,id_categoria_emitente,3333,2013(1),Médio Produtor Rural
2,operacao,id_categoria_emitente,4444,2013(1),Grande Produtor Rural
3,operacao,id_categoria_emitente,5555,2012(1)2013,Todas Categorias de Produtores Rurais
4,operacao,id_categoria_emitente,7777,2013(1)2018,Repasse a Cooperativa Singular de Crédito
5,operacao,id_categoria_emitente,8888,2013(1)2018,Repasse a Cooperativa Central de Crédito


## Etapa 2 — Diagnóstico de qualidade dos dados

Esta etapa registra evidências sobre tipos, ausências, duplicidades e inconsistências, **antes** de qualquer exclusão, substituição ou união definitiva de dados.

## 2.1 — Tipos de dados e uso de memória

**O que será feito:** Serão verificados os tipos de dados e o consumo de memória dos três arquivos utilizados no projeto.

**Como será feito:** Usaremos `DataFrame.info()` nas três tabelas; na base de operações, `show_counts=True` mostra a contagem de preenchimentos e `memory_usage="deep"` aprofunda a estimativa de memória.

**Por que será feito:** Embora as colunas já tenham sido identificadas anteriormente, precisamos verificar se o Pandas interpretou corretamente os dados numéricos, textuais e as datas.

Também será analisado o consumo de memória para avaliar o desempenho e os recursos necessários durante o processamento.

In [7]:
operacao_traduzida.info(show_counts=True, memory_usage="deep")

empreendimento.info(memory_usage="deep")

dicionario.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2442343 entries, 0 to 2442342
Data columns (total 15 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   ano_emissao                         2442343 non-null  int64  
 1   mes_emissao                         2442343 non-null  int64  
 2   data_emissao                        2442343 non-null  object 
 3   data_vencimento                     2442343 non-null  object 
 4   sigla_uf                            2442343 non-null  object 
 5   id_referencia_bacen                 2442343 non-null  int64  
 6   numero_ordem                        2442343 non-null  int64  
 7   id_categoria_emitente               2442343 non-null  int64  
 8   id_empreendimento                   2442343 non-null  int64  
 9   id_fonte_recurso                    2442343 non-null  int64  
 10  id_instrumento_credito              2442343 non-null  int64  
 11  valor_parce

### 2.1.1 — Interpretação dos resultados

A inspeção das três bases permitiu identificar
suas características estruturais.

**Base de operações:**
- 2.442.343 registros e 15 colunas.
- Consumo de memória de 673,1 MB.
- 8 colunas inteiras, 4 decimais e 3 do tipo object.
- Foram identificadas 3 colunas com valores ausentes.
- As datas estão armazenadas como object.

**Base de empreendimentos:**
- 1.050 registros e 10 colunas.
- Consumo de memória de 674,2 KB.
- Foram identificadas 2 colunas com valores ausentes.
- As datas também estão armazenadas como object.

**Base de dicionário:**
- 52 registros e 5 colunas.
- Consumo de memória de 16,7 KB.
- Não foram encontrados valores nulos.

**Conclusão preliminar:**

As bases apresentam problemas de preenchimento e
colunas de datas armazenadas como object.

A ausência de valores não será tratada automaticamente
como erro, pois pode representar uma característica
dos registros.



### 2.2 — Diagnóstico de valores ausentes

**O que será feito:** Será calculada a quantidade e o percentual de valores ausentes em cada coluna da base de operações.

**Como será feito:** Usaremos `isna().sum()` para contar ausentes, `isna().mean() * 100` para calcular os percentuais e `pd.DataFrame()` para reunir os resultados.

**Por que será feito:** Esse procedimento permite identificar quais variáveis possuem problemas de preenchimento e avaliar a dimensão
dessas ausências.

Os resultados servirão de referência para justificar as decisões de tratamento na etapa de limpeza.

In [8]:

faltantes = operacao.isna().sum()
percentual = operacao.isna().mean() * 100

diagnostico_faltantes = pd.DataFrame({
    "Quantidade de ausentes": faltantes,
    "Percentual (%)": percentual.round(2)
})


display(diagnostico_faltantes)

,Quantidade de ausentes,Percentual (%)
ano_emissao,0,0.00
mes_emissao,0,0.00
data_emissao,0,0.00
data_vencimento,0,0.00
sigla_uf,0,0.00
id_referencia_bacen,0,0.00
numero_ordem,0,0.00
id_categoria_emitente,0,0.00
id_empreendimento,0,0.00
id_fonte_recurso,0,0.00


### 2.2.1 — Interpretação dos valores ausentes em operações

**O que foi feito:** calculamos, para cada coluna de `operacao`, a quantidade e o percentual de valores nulos.

**Por que foi feito:** dimensionar a ausência de informações sem excluir registros. A inspeção aponta lacunas em `valor_quantidade_itens_financiados`, `valor_recurso_proprio` e `valor_receita_bruta_esperada`. Não podemos assumir que ausência significa zero.

### 2.2.2 — Valores ausentes na tabela de empreendimentos

**O que será feito:** calcular a quantidade e o percentual de dados ausentes em cada coluna de `empreendimento` usando `isna()`, `sum()` e `mean()`.

**Como será feito:** Usaremos `empreendimento.isna().sum()` e `.isna().mean() * 100` e exibiremos a tabela de diagnóstico.

**Por que será feito:** identificar quais informações da tabela auxiliar estão incompletas. Isso é relevante porque ela será relacionada às operações para interpretar finalidade, atividade e produto. Nenhum registro será alterado.

In [9]:
faltantes_empreendimento = empreendimento.isna().sum()
percentual_empreendimento = empreendimento.isna().mean() * 100

diagnostico_empreendimento = pd.DataFrame({
    "Quantidade de ausentes": faltantes_empreendimento,
    "Percentual (%)": percentual_empreendimento.round(2)
})

display(diagnostico_empreendimento)

,Quantidade de ausentes,Percentual (%)
id_empreendimento,0,0.00
data_inicio,0,0.00
data_fim,1019,97.05
finalidade,0,0.00
atividade,0,0.00
modalidade,0,0.00
produto,0,0.00
variedade,0,0.00
zoneamento,456,43.43
unidade_medida,0,0.00


### 2.2.3 — Valores ausentes na tabela de dicionário

**O que será feito:** verificar a quantidade e o percentual de valores ausentes em cada coluna de `dicionario`.

**Como será feito:** Usaremos `dicionario.isna().sum()` e `.isna().mean() * 100` para medir faltantes e apresentar o diagnóstico.

**Por que será feito:** conferir a completude dos códigos e das descrições que serão usados posteriormente nos relacionamentos (`merge`). Mesmo uma tabela pequena precisa ser verificada, para não introduzir descrições ausentes durante a análise.

In [10]:
faltantes_dicionario = dicionario.isna().sum()
percentual_dicionario = dicionario.isna().mean() * 100

diagnostico_dicionario = pd.DataFrame({
    "Quantidade de ausentes": faltantes_dicionario,
    "Percentual (%)": percentual_dicionario.round(2)
})

display(diagnostico_dicionario)

,Quantidade de ausentes,Percentual (%)
id_tabela,0,0.0
nome_coluna,0,0.0
chave,0,0.0
cobertura_temporal,0,0.0
valor,0,0.0


### 2.2.4 — Interpretação geral dos valores ausentes

**O que foi identificado na inspeção anterior (`info()`):**

- **Operações:** as colunas `valor_quantidade_itens_financiados`, `valor_recurso_proprio` e `valor_receita_bruta_esperada` possuem dados ausentes.
- **Empreendimentos:** `data_fim` tem 31 registros preenchidos entre 1.050, e `zoneamento` tem 594; as demais colunas aparecem integralmente preenchidas.
- **Dicionário:** as cinco colunas aparecem integralmente preenchidas.

**Interpretação:** a falta de preenchimento pode decorrer de regras de negócio ou da natureza das variáveis. Não é correto substituir valores nulos por zero nem excluir registros sem investigar seu significado e o impacto sobre as perguntas de pesquisa.



## 2.3 — Identificação de registros duplicados

### 2.3.1 — Verificação de linhas duplicadas

**O que será feito:** Será verificada a existência de registros completamente duplicados nas três bases de dados: operações,
empreendimentos e dicionário.

**Como será feito:** Usaremos `duplicated().sum()` em cada DataFrame para contar linhas integralmente repetidas, considerando todas as colunas.

**Por que será feito:** Registros duplicados podem prejudicar a qualidade dos dados e provocar contagens ou somatórios incorretos.

A identificação dessas ocorrências permite avaliar se existem problemas de repetição antes da limpeza.

**Método utilizado:** O método `duplicated()` do Pandas identifica linhas repetidas, considerando todas as colunas.

Por padrão, a primeira ocorrência é considerada original e as demais são marcadas como duplicadas.

In [11]:
duplicados_operacao = operacao.duplicated().sum()

duplicados_empreendimento = empreendimento.duplicated().sum()

duplicados_dicionario = dicionario.duplicated().sum()

print("Registros duplicados nas operações:", duplicados_operacao)

print("Registros duplicados nos empreendimentos:", duplicados_empreendimento)

print("Registros duplicados no dicionário:", duplicados_dicionario)

Registros duplicados nas operações: 0
Registros duplicados nos empreendimentos: 0
Registros duplicados no dicionário: 0


### 2.3.2 — Interpretação dos registros duplicados

**O que foi identificado:**  A verificação com o método `duplicated()` não identificou linhas completamente duplicadas em nenhuma das três bases.

- Operações: 0 registros duplicados.
- Empreendimentos: 0 registros duplicados.
- Dicionário: 0 registros duplicados.

**Conclusão preliminar:** Não foram identificadas duplicações considerando todas as colunas de cada tabela.

Entretanto, esse resultado não garante que os identificadores sejam únicos.

### 2.3.3 — Verificação dos identificadores das operações

**O que será feito:** Será verificado se existem registros com a mesma combinação de `id_referencia_bacen` e `numero_ordem`.

**Como será feito:** Usaremos `operacao.duplicated(subset=["id_referencia_bacen", "numero_ordem"]).sum()` para contar repetições da combinação.

**Por que será feito:** A ausência de linhas completamente duplicadas não garante que os identificadores sejam únicos.

Essa verificação ajuda a identificar possíveis repetições que precisam ser investigadas antes dos relacionamentos entre tabelas.

In [12]:
duplicados_chave_operacao = operacao.duplicated(
    subset=["id_referencia_bacen", "numero_ordem"]
).sum()

print(
    "Repetições da combinação de identificadores:",
    duplicados_chave_operacao
)

Repetições da combinação de identificadores: 0


### 2.3.4 — Interpretação da unicidade dos identificadores

A verificação da combinação das colunas `id_referencia_bacen` e `numero_ordem` não identificou registros repetidos.

Isso demonstra que essa combinação é única na base de operações analisada.

Portanto, ela pode ser utilizada como chave composta para identificar cada registro.

Essa característica será importante para verificar a integridade das operações e evitar contagens duplicadas.

### 2.3.5 — Verificação das chaves das tabelas auxiliares

**O que será feito:** Será verificada a existência de identificadores repetidos nas tabelas de empreendimentos e dicionário.

Na tabela de empreendimentos será utilizada a coluna `id_empreendimento`.

No dicionário será analisada a combinação de `id_tabela`, `nome_coluna` e `chave`.

**Como será feito:** Usaremos `duplicated(subset=...)` para conferir `id_empreendimento` e a combinação `id_tabela`, `nome_coluna` e `chave` no dicionário.

**Por que será feito:** Identificadores repetidos podem provocar multiplicação de registros durante operações de relacionamento
entre tabelas (merge).

A verificação permite avaliar se essas combinações podem ser usadas como chaves nas próximas etapas.

In [13]:
duplicados_chave_empreendimento = empreendimento.duplicated(
    subset=["id_empreendimento"]
).sum()


duplicados_chave_dicionario = dicionario.duplicated(
    subset=["id_tabela", "nome_coluna", "chave"]
).sum()

print(
    "Identificadores repetidos em empreendimentos:",
    duplicados_chave_empreendimento
)

print(
    "Combinações repetidas no dicionário:",
    duplicados_chave_dicionario
)

Identificadores repetidos em empreendimentos: 0
Combinações repetidas no dicionário: 0


### 2.3.6 — Interpretação das chaves das tabelas auxiliares

**Resultados encontrados:** A verificação de duplicidade dos identificadores das tabelas auxiliares apresentou os seguintes resultados:

- Empreendimentos: 0 identificadores repetidos.
- Dicionário: 0 combinações repetidas.

**Interpretação:** Os resultados indicam que `id_empreendimento` é único na tabela de empreendimentos.

No dicionário, a combinação de `id_tabela`, `nome_coluna` e `chave` também é única.

Essas características são importantes para evitar a multiplicação indevida de registros nos relacionamentos entre tabelas.

**Conclusão:** Não foram identificadas duplicidades nas chaves examinadas.


## 2.4 — Categorias inconsistentes e valores inválidos

### 2.4.1 — Verificação das categorias de finalidade

**O que será feito:** Serão identificadas as categorias existentes na coluna `finalidade` da tabela de empreendimentos, juntamente com a quantidade de ocorrências de cada categoria.

**Como será feito:** Usaremos `empreendimento["finalidade"].value_counts(dropna=False)` para contar todas as categorias, incluindo valores ausentes.

**Por que será feito:** Essa verificação permite identificar possíveis inconsistências na classificação dos financiamentos, como diferenças de escrita, espaços desnecessários, letras maiúsculas e minúsculas ou categorias vazias.

**Método utilizado:** O método `value_counts()` do Pandas será utilizado para contar a frequência de cada categoria.

O parâmetro `dropna=False` inclui os valores ausentes na contagem.

In [14]:
contagem_finalidade = empreendimento["finalidade"].value_counts(
    dropna=False
)

print(contagem_finalidade)

finalidade
Custeio             492
Investimento        331
Comercialização     168
Industrialização     59
Name: count, dtype: int64


### 2.4.2 — Interpretação das categorias de finalidade

**Resultados encontrados:** A tabela de empreendimentos apresenta quatro categorias de finalidade:

- Custeio: 492 registros.
- Investimento: 331 registros.
- Comercialização: 168 registros.
- Industrialização: 59 registros.

**Interpretação:** A categoria Custeio possui a maior quantidade de registros na tabela de empreendimentos.

Não foram observadas diferenças evidentes de escrita ou valores nulos nessa contagem.

Entretanto, a contagem não permite afirmar qual finalidade recebeu o maior volume de crédito.

**Conclusão preliminar:** As categorias apresentam uma padronização aparente.
Ainda será necessário investigar espaços ocultos e outras possíveis inconsistências.

### 2.4.3 — Verificação da padronização das finalidades

**O que será feito:** Serão comparados os valores originais da coluna `finalidade` com versões padronizadas temporariamente.

**Como será feito:** Usaremos os métodos de texto `.str.strip()` e `.str.casefold()` para comparar temporariamente os valores originais e normalizados, sem substituir a coluna.

**Por que será feito:** Essa comparação permite identificar espaços extras e diferenças de letras maiúsculas e minúsculas que possam prejudicar agrupamentos e análises.

Nenhuma alteração será realizada nos dados originais.

In [15]:
original = empreendimento["finalidade"]

padronizada = original.str.strip().str.casefold()

inconsistentes = (
    original.str.casefold() != padronizada
).sum()

print("Registros com espaços extras:", inconsistentes)

print(
    "Quantidade de categorias após padronização:",
    padronizada.nunique()
)

Registros com espaços extras: 0
Quantidade de categorias após padronização: 4


### 2.4.4 — Interpretação da padronização das finalidades

**Resultados encontrados:** 

- Registros com espaços extras: 0.
- Quantidade de categorias após padronização: 4.

**Interpretação:** Não foram identificados espaços extras nas categorias da coluna `finalidade`.

A quantidade de categorias permaneceu igual após a padronização, indicando que não foram encontradas variações de escrita que alterassem essa contagem.

**Conclusão preliminar:** A coluna `finalidade` apresenta consistência nas verificações realizadas, não sendo necessária uma correção neste momento.

### 2.4.5 — Verificação das demais categorias

**O que será feito:** Serão verificadas as categorias existentes nas colunas `atividade`, `modalidade` e `produto` da tabela de
empreendimentos.

**Como será feito:** Percorreremos `atividade`, `modalidade` e `produto` com `for`, comparando valores originais e versões produzidas com `.str.strip()` e `.str.casefold()`; também contaremos categorias com `nunique()`.

**Por que será feito:** Essa análise permite identificar diferenças de escrita, espaços desnecessários e possíveis inconsistências
que possam prejudicar agrupamentos futuros.

Nenhum dado será alterado nesta etapa.

In [16]:
colunas = ["atividade", "modalidade", "produto"]

for coluna in colunas:

    original = empreendimento[coluna]
    padronizada = original.str.strip().str.casefold()

    print(f"\nColuna: {coluna}")

    print("Categorias originais:", original.nunique())

    print("Categorias padronizadas:", padronizada.nunique())

    print(
        "Registros com espaços extras:",
        (original.str.casefold() != padronizada).sum()
    )

    print("\nCategorias mais frequentes:")
    print(original.value_counts(dropna=False).head(10))


Coluna: atividade
Categorias originais: 2
Categorias padronizadas: 2
Registros com espaços extras: 0

Categorias mais frequentes:
atividade
Agrícola       842
Pecuário(a)    208
Name: count, dtype: int64

Coluna: modalidade
Categorias originais: 41
Categorias padronizadas: 41
Registros com espaços extras: 0

Categorias mais frequentes:
modalidade
LAVOURA                                               417
FORMAÇÃO DE CULTURAS PERENES                           97
MÁQUINAS, EQUIPAMENTOS, MATERIAIS E UTENSÍLIOS         79
ESTOCAGEM                                              63
BENEFICIAMENTO OU INDUSTRIALIZAÇÃO                     55
FGPP-Financiamento para Garantia de Preços ao Prod     47
SERVIÇOS PROFISSIONAIS/TÉCNICOS                        38
FEE (EX-LEC)                                           30
AQUISIÇÃO DE ATIVOS OPERACIONAIS                       23
AQUISIÇÃO DE ANIMAIS                                   21
Name: count, dtype: int64

Coluna: produto
Categorias originais: 316
C

### 2.4.6 — Interpretação das demais categorias

**Resultados encontrados:** A análise das colunas atividade, modalidade e produto apresentou os seguintes resultados:

- Atividade: 2 categorias distintas.
- Modalidade: 41 categorias distintas.
- Produto: 316 categorias distintas.
- Não foram identificados espaços extras.
- A quantidade de categorias permaneceu igual
  após a padronização.

**Interpretação:** A tabela apresenta duas atividades principais:
Agrícola e Pecuário(a).

As colunas modalidade e produto possuem maior diversidade de classificações.

Nas verificações realizadas, não foram identificadas inconsistências de espaços ou diferenças de escrita que reduzissem a quantidade de categorias após
a padronização.

**Conclusão preliminar:** As categorias apresentam consistência nas verificações realizadas.

Entretanto, ainda será necessário investigar possíveis diferenças de nomenclatura e a correspondência dos códigos entre as tabelas.

### 2.4.7 — Verificação de valores financeiros

**O que será feito:** Será identificada a quantidade de valores negativos e iguais a zero nas colunas financeiras da base
de operações de crédito rural.

**Como será feito:** Percorreremos três colunas financeiras com `for`; contaremos negativos com `< 0`, zeros com `== 0` e ausentes com `isna().sum()`.

**Por que será feito:** Valores negativos ou iguais a zero podem representar situações que precisam ser investigadas conforme
o significado de cada variável.

Essa verificação permite identificar possíveis valores inválidos antes da limpeza dos dados.

Nenhum registro será alterado ou excluído.

In [17]:
colunas_valores = [
    "valor_parcela_credito",
    "valor_recurso_proprio",
    "valor_receita_bruta_esperada"
]


for coluna in colunas_valores:

    print(f"\nColuna: {coluna}")

    print("Valores negativos:", (operacao[coluna] < 0).sum())

    print("Valores iguais a zero:", (operacao[coluna] == 0).sum())

    print("Valores ausentes:", operacao[coluna].isna().sum())


Coluna: valor_parcela_credito
Valores negativos: 0
Valores iguais a zero: 0
Valores ausentes: 0

Coluna: valor_recurso_proprio
Valores negativos: 0
Valores iguais a zero: 541844
Valores ausentes: 1720391

Coluna: valor_receita_bruta_esperada
Valores negativos: 0
Valores iguais a zero: 0
Valores ausentes: 1652191


### 2.4.8 — Interpretação dos valores financeiros

**Resultado registrado na execução anterior:**

| Coluna | Negativos | Zeros | Ausentes |
|---|---:|---:|---:|
| `valor_parcela_credito` | 0 | 0 | 0 |
| `valor_recurso_proprio` | 0 | 541.844 | 1.720.391 |
| `valor_receita_bruta_esperada` | 0 | 0 | 1.652.191 |

**Interpretação:** não foram encontrados valores negativos nas três colunas examinadas. Na coluna `valor_recurso_proprio`, o valor zero foi registrado em parte das linhas e muitas outras estão sem informação (`NaN`); **zero e ausente não são equivalentes**. Não se pode concluir, apenas dessa contagem, se os ausentes decorrem de regra de preenchimento, tipo de operação ou problema de qualidade.

**Decisão nesta etapa:** manter os dados originais, sem imputar ou excluir registros. Os significados e a relevância das ausências serão investigados na etapa de limpeza e transformação. Os números desta tabela são resultados salvos da execução anterior e devem ser conferidos se os CSVs forem substituídos.
